# Fine-tune LLM with PyTorch FSDP and QLora on Amazon SageMaker AI using ModelTrainer

In this notebook, we fine-tune LLM on Amazon SageMaker AI, using Python scripts and SageMaker ModelTrainer for executing a training job.

## Prerequisites

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade

***

In [1]:
import os

os.environ["AWS_PROFILE"] = "bpistone-dev-account-role"

## Setup Configuration file path

In [2]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

In [3]:
sagemaker_session = Session()
sagemaker_session_bucket = sagemaker_session.default_bucket()
sagemaker_session = Session(default_bucket=sagemaker_session_bucket)

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sagemaker_session.boto_region_name}")

[06/03/26 14:37:41] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=794156;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=943463;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/bpistone/Library/Application Support/sagemaker/config.yaml


[06/03/26 14:37:43] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=484137;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=471604;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[06/03/26 14:37:44] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=861115;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=127990;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[06/03/26 14:37:45] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=589086;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=479746;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker role arn: arn:aws:iam::691148928602:role/mlops-sagemaker-execution-role
sagemaker bucket: sagemaker-us-east-1-691148928602
sagemaker session region: us-east-1


If you have created a Managed MLflow server, copy the `ARN` code here and assign a name to the experiment

In [4]:
import os

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

os.environ["HF_TOKEN"] = "<HF_TOKEN>"
os.environ["model_id"] = model_id
os.environ["mlflow_uri"] = ""
os.environ["mlflow_experiment_name"] = "mistral-7b-v03-reasoning-multi-language"

***

## Visualize and upload the dataset

We are going to load [HuggingFaceH4/Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) dataset

In [5]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")

dataset

[06/03/26 14:37:47] INFO     HTTP Request: HEAD                                                     ]8;id=120733;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=860762;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking/re                
                             solve/main/README.md "HTTP/1.1 307 Temporary Redirect"                                

                    INFO     HTTP Request: HEAD                                                     ]8;id=398841;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=161943;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/datasets/HuggingFaceH4/Multil                
                             ingual-Thinking/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7/README.md                    
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: HEAD                                                     ]8;id=685695;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=688870;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking/re                
                             solve/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7/Multilingual-Thinking.p                
                             y "HTTP/1.1 404 Not Found"                                                            

[06/03/26 14:37:48] INFO     HTTP Request: HEAD                                                     ]8;id=881660;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=160914;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Hug                
                             gingFaceH4/Multilingual-Thinking/HuggingFaceH4/Multilingual-Thinking.p                
                             y "HTTP/1.1 404 Not Found"                                                            

                    INFO     HTTP Request: GET                                                      ]8;id=626435;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=301891;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/datasets/HuggingFaceH4/Multilingual-Thinkin                
                             g/revision/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7 "HTTP/1.1 200 OK"                 

...

                    INFO     HTTP Request: GET                                                      ]8;id=45692;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=378937;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://datasets-server.huggingface.co/info?dataset=HuggingFaceH4/Mult                
                             ilingual-Thinking "HTTP/1.1 200 OK"                                                   

                    INFO     HTTP Request: GET                                                      ]8;id=756520;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=890329;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/datasets/HuggingFaceH4/Multilingual-Thinkin                
                             g/tree/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7/data?recursive=true&ex                
                             pand=false "HTTP/1.1 200 OK"                                                          

                    INFO     HTTP Request: GET                                                      ]8;id=66116;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=133648;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/datasets/HuggingFaceH4/Multilingual-Thinkin                
                             g/tree/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7?recursive=false&expand                
                             =false "HTTP/1.1 200 OK"                                                              

                    INFO     HTTP Request: HEAD                                                     ]8;id=232115;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=849282;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking/re                
                             solve/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7/dataset_infos.json                     
                             "HTTP/1.1 404 Not Found"                                                              

Dataset({
    features: ['reasoning_language', 'developer', 'user', 'analysis', 'final', 'messages'],
    num_rows: 1000
})

In [6]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

,reasoning_language,developer,user,analysis,final,messages
0,French,You are an AI chatbot with a lively and energe...,Can you show me the latest trends on Twitter r...,"D'accord, l'utilisateur demande les tendances ...",Hey there! While I can't check Twitter (X) in...,[{'content': 'reasoning language: French You ...
1,English,You are an intelligent assistant that can answ...,Can you provide me with a list of the top-rate...,"\nOkay, the user is asking for the top-rated s...",Netflix does not publicly release real-time li...,[{'content': 'reasoning language: English You...
2,German,"Always refuse to answer, responding simply 'No'",Can you check how many followers I currently h...,"In Ordnung, der Benutzer fragt, ob ich seine T...",No.,[{'content': 'reasoning language: German Alwa...
3,Spanish,You are an AI that formats its responses in si...,I'd like to plan a trip to Rome for 7 days. I ...,"Perfecto, veamos. El usuario quiere un viaje d...","**Rome 7-Day Itinerary: History, Food, & Relax...",[{'content': 'reasoning language: Spanish You...
4,English,You are a formal and professional chatbot. You...,I've been feeling quite low lately. Could you ...,"\nOkay, the user is feeling down and wants act...",I'm truly sorry to hear you're feeling this wa...,[{'content': 'reasoning language: English You...


In [7]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.1, random_state=42)
train, test = train_test_split(train, test_size=10, random_state=42)

print("Number of train elements: ", len(train))
print("Number of test elements: ", len(val))
print("Number of test elements: ", len(test))

Number of train elements:  890
Number of test elements:  100
Number of test elements:  10


Create a prompt template and load the dataset with a random sample to try summarization.

In [8]:
import textwrap
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(model_id)

# Define PAD token
if tokenizer.pad_token is None:
    # Define PAD token
    tokenizer.pad_token = tokenizer.eos_token


def prepare_dataset(sample):

    messages = []

    first_user_message = True
    for el in sample["messages"]:
        if el["role"] == "system":
            system_prompt = """
            You are an AI assistant that thinks in {language} but responds in English.

            IMPORTANT: Follow this exact format for every response:
            1. First, write your reasoning and thoughts inside <think>...</think> tags
            2. Then, provide your final answer in English

            Always think through the problem in {language}, then translate your conclusion to English for the final response.
            """

            system_prompt = system_prompt.format(language=sample["reasoning_language"])
            system_prompt = textwrap.dedent(system_prompt).strip()
        elif el["role"] == "user":
            if first_user_message:
                first_user_message = False
                messages.append(
                    {"role": "user", "content": system_prompt + "\n\n" + el["content"]}
                )
            else:
                messages.append({"role": "user", "content": el["content"]})
        else:
            if (
                el["thinking"] is not None
                and el["thinking"] != ""
                and el["thinking"] != "null"
            ):
                messages.append(
                    {
                        "role": "assistant",
                        "content": f"<think>\n{el["thinking"]}\n</think>\n{el["content"]}",
                    }
                )
            else:
                messages.append(
                    {
                        "role": "assistant",
                        "content": f"{el["content"]}",
                    }
                )

    # Apply chat template
    sample["text"] = tokenizer.apply_chat_template(messages, tokenize=False)

    return sample

[06/03/26 14:37:56] INFO     HTTP Request: HEAD                                                     ]8;id=96285;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=60140;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3/resolve/main                
                             /config.json "HTTP/1.1 307 Temporary Redirect"                                        

                    INFO     HTTP Request: HEAD                                                     ]8;id=184351;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=395870;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-I                
                             nstruct-v0.3/c170c708c41dac9275d15a8fff4eca08d52bab71/config.json                     
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: HEAD                                                     ]8;id=404611;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=314630;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3/resolve/main                
                             /tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"                              

                    INFO     HTTP Request: HEAD                                                     ]8;id=884020;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=624690;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-I                
                             nstruct-v0.3/c170c708c41dac9275d15a8fff4eca08d52bab71/tokenizer_config                
                             .json "HTTP/1.1 200 OK"                                                               

                    INFO     HTTP Request: HEAD                                                     ]8;id=295386;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=688071;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3/resolve/main                
                             /tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"                              

                    INFO     HTTP Request: HEAD                                                     ]8;id=471797;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=670091;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-I                
                             nstruct-v0.3/c170c708c41dac9275d15a8fff4eca08d52bab71/tokenizer_config                
                             .json "HTTP/1.1 200 OK"                                                               

                    INFO     HTTP Request: GET                                                      ]8;id=754079;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=710980;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/models/mistralai/Mistral-7B-Instruct-v0.3/t                
                             ree/main/additional_chat_templates?recursive=false&expand=false                       
                             "HTTP/1.1 404 Not Found"                                                              

                    INFO     HTTP Request: GET                                                      ]8;id=228712;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=980864;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/models/mistralai/Mistral-7B-Instruct-v0.3/t                
                             ree/main?recursive=true&expand=false "HTTP/1.1 200 OK"                                

In [9]:
from datasets import Dataset, DatasetDict
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

dataset = DatasetDict({"train": train_dataset, "val": val_dataset})

train_dataset = dataset["train"].map(
    prepare_dataset, remove_columns=list(train_dataset.features)
)

print(train_dataset[randint(0, len(dataset))]["text"])

val_dataset = dataset["val"].map(
    prepare_dataset, remove_columns=list(val_dataset.features)
)

Map:   0%|          | 0/890 [00:00<?, ? examples/s]

<s>[INST] You are an AI assistant that thinks in German but responds in English.

IMPORTANT: Follow this exact format for every response:
1. First, write your reasoning and thoughts inside <think>...</think> tags
2. Then, provide your final answer in English

Always think through the problem in German, then translate your conclusion to English for the final response.

Draft an email to my boss, Mr. Johnson, subject line 'Project Update'. In the email, let him know that we've completed 75% of the project tasks, and we're ahead of schedule. Attach the file named 'ProjectUpdate.pdf' from my desktop. Also, set a polite and professional tone.[/INST] <think>
Gut, lass mich das mal überlegen. Der Benutzer möchte, dass ich eine E-Mail an Herrn Johnson verfasse mit dem Betreff „Projektaktualisierung“. Er soll wissen, dass 75 % der Projektarbeiten abgeschlossen sind und dass man sich im Zeitplan voraus bewegt. Außerdem soll eine PDF-Datei namens „ProjectUpdate.pdf“ vom Desktop angehängt werden. 

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

### Upload to Amazon S3

In [10]:
import boto3
import shutil
from sagemaker.core.helper.session_helper import Session

In [11]:
sagemaker_session = Session()
s3_client = boto3.client('s3')

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [12]:
# save train_dataset to s3 using our SageMaker session
if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-modeltrainer-dsz3"
else:
    input_path = f"datasets/llm-fine-tuning-modeltrainer-dsz3"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.json"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.json"

In [13]:
# Save datasets to s3
# We will fine tune only with 20 records due to limited compute resource for the workshop
train_dataset.to_json("./data/train/dataset.json", orient="records")
val_dataset.to_json("./data/val/dataset.json", orient="records")

s3_client.upload_file("./data/train/dataset.json", bucket_name, f"{input_path}/train/dataset.json")
s3_client.upload_file("./data/val/dataset.json", bucket_name, f"{input_path}/val/dataset.json")

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Training data uploaded to:
s3://sagemaker-us-east-1-691148928602/datasets/llm-fine-tuning-modeltrainer-dsz3/train/dataset.json
s3://sagemaker-us-east-1-691148928602/datasets/llm-fine-tuning-modeltrainer-dsz3/val/dataset.json


***

## Model fine-tuning

We are now ready to fine-tune our model. We will use the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) from transfomers to fine-tune our model. We prepared a script [train.py](./scripts/train.py) which will loads the dataset from disk, prepare the model, tokenizer and start the training.

### Training configurations

For configuration we use `TrlParser`, that allows us to provide hyperparameters in a `yaml` file. This yaml will be uploaded and provided to Amazon SageMaker similar to our datasets. We are saving the config file as `args.yaml` and upload it to S3.

In [ ]:
%%bash

cat > ./args.yaml <<EOF
model_id: "${model_id}"                           # Hugging Face model id
mlflow_uri: "${mlflow_uri}"                       # MLflow tracking server URI
mlflow_experiment_name: "${mlflow_experiment_name}" # MLflow experiment name
# sagemaker specific parameters
output_dir: "/opt/ml/model"                       # path to where SageMaker will upload the model 
checkpoint_dir: "/opt/ml/checkpoints/"            # directory for saving training checkpoints
train_dataset_path: "/opt/ml/input/data/train/"   # path to where S3 saves train dataset
val_dataset_path: "/opt/ml/input/data/val/"       # path to where S3 saves test dataset
token: "${HF_TOKEN}"                              # Hugging Face API token
merge_weights: true                               # merge weights in the base model
# training parameters
apply_truncation: true                           # apply truncation to datasets
attn_implementation: "flash_attention_2"         # attention implementation type
learning_rate: 2e-5                              # learning rate scheduler
num_train_epochs: 10                             # number of training epochs
per_device_train_batch_size: 1                   # batch size per device during training
per_device_eval_batch_size: 2                    # batch size for evaluation
gradient_accumulation_steps: 16                  # number of steps before performing a backward/update pass
gradient_checkpointing: true                     # use gradient checkpointing
torch_dtype: "bfloat16"                          # float precision type
bf16: true                                       # use bfloat16 precision
tf32: true                                       # use tf32 precision
ignore_data_skip: true                           # skip data loading errors
logging_strategy: "steps"                        # logging strategy
logging_steps: 1                                 # log every N steps
log_on_each_node: false                          # disable logging on each node
ddp_find_unused_parameters: false                # DDP unused parameter detection
save_total_limit: 1                              # maximum number of checkpoints to keep
save_steps: 100                                  # Save checkpoint every this many steps
warmup_steps: 50                                 # number of warmup steps
weight_decay: 0.01                               # weight decay coefficient
dataloader_pin_memory: false                     # pin memory for dataloader
# LoRA parameters
load_in_4bit: false                              # enable 4-bit quantization
lora_r: 16                                       # LoRA rank
lora_alpha: 32                                   # LoRA alpha parameter
lora_dropout: 0.1                                # LoRA dropout rate
EOF

Lets upload the config file to S3.

In [15]:
import os

if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-modeltrainer-dsz3"
else:
    input_path = f"datasets/llm-fine-tuning-modeltrainer-dsz3"

train_config_s3_path = f"s3://{bucket_name}/{input_path}/config/args.yaml"

# upload the model yaml file to s3
model_yaml = "args.yaml"
s3_client.upload_file(model_yaml, bucket_name, f"{input_path}/config/args.yaml")

print(f"Training config uploaded to:")
print(train_config_s3_path)

Training config uploaded to:
s3://sagemaker-us-east-1-691148928602/datasets/llm-fine-tuning-modeltrainer-dsz3/config/args.yaml


### DeepSpeed configurations

In [16]:
%%bash

cat > ./accelerate_config.yaml <<EOF
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  deepspeed_multinode_launcher: standard
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: true
  zero3_save_16bit_model: true
  zero_stage: 3
distributed_type: DEEPSPEED
downcast_bf16: 'no'
main_training_function: main
mixed_precision: bf16
rdzv_backend: c10d                        # static for single node, c10d for single and multi-node
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
EOF

Lets upload the config file to S3.

In [17]:
import os

if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-modeltrainer-dsz3"
else:
    input_path = f"datasets/llm-fine-tuning-modeltrainer-dsz3"

train_accelerate_config_s3_path = f"s3://{bucket_name}/{input_path}/accelerate_config/accelerate_config.yaml"

# upload the model yaml file to s3
model_yaml = "accelerate_config.yaml"
s3_client.upload_file(model_yaml, bucket_name, f"{input_path}/accelerate_config/accelerate_config.yaml")

print(f"Accelerate config uploaded to:")
print(train_accelerate_config_s3_path)

Accelerate config uploaded to:
s3://sagemaker-us-east-1-691148928602/datasets/llm-fine-tuning-modeltrainer-dsz3/accelerate_config/accelerate_config.yaml


## Fine-tune model

Below estimtor will train the model with QLoRA, merge the adapter in the base model and save in S3

#### Get PyTorch image_uri

We are going to use the native PyTorch container image, pre-built for Amazon SageMaker

In [18]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session

[06/03/26 14:38:03] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=379709;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=972176;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

In [19]:
sagemaker_session = Session()

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [20]:
instance_type = "ml.g5.12xlarge"
instance_count = 1

instance_type

'ml.g5.12xlarge'

In [21]:
image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_session.region_name,
    version="2.8.0",
    instance_type=instance_type,
    image_scope="training"
)

image_uri

[06/03/26 14:38:05] INFO     Defaulting to only available Python version: py312                   ]8;id=664899;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=909229;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

'763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.8.0-gpu-py312'

In [22]:
from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.model_trainer import ModelTrainer

args = [
    "--entrypoint",
    "train.py",
    "--accelerate_config",
    "/opt/ml/input/data/accelerate_config/accelerate_config.yaml",
    "--config",
    "/opt/ml/input/data/config/args.yaml",
]


# Define the script to be run
source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    command=f"bash sm_accelerate_train.sh {' '.join(args)}",
)

# Define the compute
compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    keep_alive_period_in_seconds=1800,
)

# define Training Job Name
job_name = f"train-{model_id.split('/')[-1].replace('.', '-')}-dsz3"

# define OutputDataConfig path
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

# Define the ModelTrainer
model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=18000),
    output_data_config=OutputDataConfig(
        s3_output_path=output_path, compression_type="NONE"
    ),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoint", local_path="/opt/ml/checkpoints"
    ),
)

/var/folders/t6/cdfh9jv54zb2mfrvk6vq_tmc0000gq/T/ipykernel_38042/3922448487.py:1: DeprecationWarning: sagemaker.train.configs has been moved to sagemaker.core.training.configs. Please update your imports. This shim will be removed in a future version.
  from sagemaker.train.configs import (


[06/03/26 14:38:06] INFO     SageMaker session not provided. Using default Session.                  ]8;id=274848;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=653509;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

[06/03/26 14:38:07] INFO     Role not provided. Using default role:                                  ]8;id=865707;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=708913;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/train/defaults.py#75\75]8;;\
                             arn:aws:iam::691148928602:role/mlops-sagemaker-execution-role                         

                    INFO     Training image URI:                                               ]8;id=728569;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=616476;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.8                     
                             .0-gpu-py312                                                                          

In [23]:
from sagemaker.train.configs import InputData

# Pass the input data
train_input = InputData(
    channel_name="train",
    data_source=train_dataset_s3_path, # S3 path where training data is stored
)

val_input = InputData(
    channel_name="val",
    data_source=val_dataset_s3_path, # S3 path where training data is stored
)

config_input = InputData(
    channel_name="config",
    data_source=train_config_s3_path, # S3 path where training data is stored
)

accelerate_config_input = InputData(
    channel_name="accelerate_config",
    data_source=train_accelerate_config_s3_path,  # S3 path where training data is stored
)

# Check input channels configured
data = [train_input, val_input, config_input, accelerate_config_input]
data

[InputData(channel_name='train', data_source='s3://sagemaker-us-east-1-691148928602/datasets/llm-fine-tuning-modeltrainer-dsz3/train/dataset.json', content_type=None),
 InputData(channel_name='val', data_source='s3://sagemaker-us-east-1-691148928602/datasets/llm-fine-tuning-modeltrainer-dsz3/val/dataset.json', content_type=None),
 InputData(channel_name='config', data_source='s3://sagemaker-us-east-1-691148928602/datasets/llm-fine-tuning-modeltrainer-dsz3/config/args.yaml', content_type=None),
 InputData(channel_name='accelerate_config', data_source='s3://sagemaker-us-east-1-691148928602/datasets/llm-fine-tuning-modeltrainer-dsz3/accelerate_config/accelerate_config.yaml', content_type=None)]

In [24]:
# starting the train job with our uploaded datasets as input
model_trainer.train(input_data_config=data, wait=False)

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=978805;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=930262;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/bpistone/Library/Application Support/sagemaker/config.yaml


[06/03/26 14:38:14] INFO     Creating training_job resource.                                     ]8;id=231719;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=979160;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/resources.py#31116\31116]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=938865;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=184003;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-east-1                                  ]8;id=196145;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=152358;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=280458;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=68294;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[06/03/26 14:38:15] WARNING  Not displaing the training container logs as 'wait' is set to     ]8;id=228888;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=607235;file:///Users/bpistone/miniconda3/envs/python312-sm3/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#811\811]8;;\
                             False.                                                                                

***

# Model Deployment

In the following sections, we are going to deploy the fine-tuned model on an Amazon SageMaker Real-time endpoint.

## Load Fine-Tuned model

In [ ]:
import boto3
import sagemaker

In [ ]:
sagemaker_session = sagemaker.Session()

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
job_prefix = f"train-{model_id.split('/')[-1].replace('.', '-')}-dsz3"

In [ ]:
def get_last_job_name(job_name_prefix):
    sagemaker_client = boto3.client('sagemaker')

    matching_jobs = []
    next_token = None

    while True:
        # Prepare the search parameters
        search_params = {
            'Resource': 'TrainingJob',
            'SearchExpression': {
                'Filters': [
                    {
                        'Name': 'TrainingJobName',
                        'Operator': 'Contains',
                        'Value': job_name_prefix
                    },
                    {
                        'Name': 'TrainingJobStatus',
                        'Operator': 'Equals',
                        'Value': "Completed"
                    }
                ]
            },
            'SortBy': 'CreationTime',
            'SortOrder': 'Descending',
            'MaxResults': 100
        }

        # Add NextToken if we have one
        if next_token:
            search_params['NextToken'] = next_token

        # Make the search request
        search_response = sagemaker_client.search(**search_params)

        # Filter and add matching jobs
        matching_jobs.extend([
            job['TrainingJob']['TrainingJobName'] 
            for job in search_response['Results']
            if job['TrainingJob']['TrainingJobName'].startswith(job_name_prefix)
        ])

        # Check if we have more results to fetch
        next_token = search_response.get('NextToken')
        if not next_token or matching_jobs:  # Stop if we found at least one match or no more results
            break

    if not matching_jobs:
        raise ValueError(f"No completed training jobs found starting with prefix '{job_name_prefix}'")

    return matching_jobs[0]

In [ ]:
job_name = get_last_job_name(job_prefix)

job_name

#### Inference configurations

In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker import Model

In [ ]:
instance_count = 1
instance_type = "ml.g5.12xlarge"
health_check_timeout = 700

In [ ]:
image_uri = sagemaker.image_uris.retrieve(
    framework="djl-lmi",
    region=sagemaker_session.boto_session.region_name,
    version="latest"
)

image_uri = image_uri.split("/")[0] + "/djl-inference:0.33.0-lmi15.0.0-cu128"

image_uri

In [ ]:
if default_prefix:
    model_data_path=f"s3://{bucket_name}/{default_prefix}/{job_prefix}/{job_name}/output/model/"
else:
    model_data_path = f"s3://{bucket_name}/{job_prefix}/{job_name}/output/model/"

model_data = {
    "S3DataSource": {
        "S3Uri": model_data_path,
        "S3DataType": "S3Prefix",
        "CompressionType": "None",
    }
}

model = Model(
    image_uri=image_uri,
    model_data=model_data,
    role=get_execution_role(),
    env={
        "HF_MODEL_ID": "/opt/ml/model",  # path to where sagemaker stores the model
        "OPTION_TRUST_REMOTE_CODE": "true",
        "OPTION_ROLLING_BATCH": "vllm",
        "OPTION_DTYPE": "bf16",
        "OPTION_QUANTIZE": "fp8",
        "OPTION_TENSOR_PARALLEL_DEGREE": "max",
        "OPTION_MAX_ROLLING_BATCH_SIZE": "32",
        "OPTION_MODEL_LOADING_TIMEOUT": "3600"
    },
)

In [ ]:
endpoint_name = f"{model_id.split('/')[-1].replace('.', '-')}-djl"

In [ ]:
predictor = model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=instance_count,
    instance_type=instance_type,
    container_startup_health_check_timeout=health_check_timeout,
    model_data_download_timeout=3600
)

#### Predict

In [ ]:
import sagemaker

In [ ]:
sagemaker_session = sagemaker.Session()

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

endpoint_name = f"{model_id.split('/')[-1].replace('.', '-')}-djl"

In [ ]:
predictor = sagemaker.Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sagemaker_session,
    serializer=sagemaker.serializers.JSONSerializer(),
    deserializer=sagemaker.deserializers.JSONDeserializer(),
)

In [ ]:
import pandas as pd
import textwrap

eval_dataset = []

index = 1
for sample in test_dataset:

    print("Processing item ", index)

    messages = []
    message_index = 0
    for el in sample["messages"]:
        if message_index == len(sample["messages"]) - 1:
            break

        if el["role"] == "system":
            system_prompt = """
            You are an AI assistant that thinks in {language} but responds in English.

            IMPORTANT: Follow this exact format for every response:
            1. First, write your reasoning and thoughts inside <think>...</think> tags
            2. Then, provide your final answer in English

            Always think through the problem in {language}, then translate your conclusion to English for the final response.
            """

            system_prompt = system_prompt.format(language=sample["reasoning_language"])
            system_prompt = textwrap.dedent(system_prompt).strip()

            messages.append({"role": "system", "content": system_prompt})
        elif el["role"] == "user":
            messages.append({"role": "user", "content": el["content"]})
        else:
            if (
                el["thinking"] is not None
                and el["thinking"] != ""
                and el["thinking"] != "null"
            ):
                messages.append(
                    {
                        "role": "assistant",
                        "content": f"<think>\n{el["thinking"]}\n</think>\n{el["content"]}",
                    }
                )
            else:
                messages.append(
                    {
                        "role": "assistant",
                        "content": f"{el["content"]}",
                    }
                )

        message_index += 1

    response = predictor.predict(
        {
            "messages": messages,
            "max_tokens": 4096,
            "stop": ["[/INST]"],
            "temperature": 0.1,
            "top_p": 0.9,
            "repetition_penalty": 1.15,  # Add repetition penalty
            "no_repeat_ngram_size": 3,  # Prevent 3-gram repetition
            "do_sample": True,
        }
    )

    eval_dataset.append(
        [
            [el["content"] for el in messages if el["role"] == "system"][0],
            [el["content"] for el in messages if el["role"] == "user"],
            response["choices"][0]["message"]["content"],
        ]
    )

    index += 1

    print("**********************************************")

eval_dataset_df = pd.DataFrame(eval_dataset, columns=["system", "question", "answer"])

eval_dataset_df.to_json("./eval_dataset_results.jsonl", orient="records", lines=True)

#### Delete Endpoint

In [ ]:
import sagemaker

In [ ]:
sagemaker_session = sagemaker.Session()

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

endpoint_name = f"{model_id.split('/')[-1].replace('.', '-')}-djl"

In [ ]:
predictor = sagemaker.Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sagemaker_session,
    serializer=sagemaker.serializers.JSONSerializer(),
    deserializer=sagemaker.deserializers.JSONDeserializer(),
)

In [ ]:
predictor.delete_model()
predictor.delete_endpoint(delete_endpoint_config=True)